# TinyVLM: SigLIP2 + Qwen2.5-0.5B, fine-tuned for VQA

**Before running:** in the Kaggle notebook side panel, set:
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On (required to download HF models/datasets and to push to the Hub at the end)
- **Add data:** search for and add the Kaggle dataset `adityajn105/flickr8k` as an input

Pipeline:
1. Install dependencies
2. Write out the project's `.py` files
3. Prepare Stage 1 data (Flickr8k captions) and Stage 2 data (VQAv2 subset)
4. Stage 1 training -- projector pretraining
5. Stage 2 training -- LoRA instruction tuning
6. Quick inference sanity check
7. Push weights to the Hugging Face Hub


## 1. Install dependencies

In [1]:
!pip install -q transformers>=4.44.0 peft>=0.11.0 accelerate>=0.30.0 datasets huggingface_hub


## 2. Write out project files
These mirror the `model.py` / `dataset.py` / training scripts from the companion repo, embedded here so the notebook is self-contained.

In [24]:
%%writefile model.py
"""
model.py

Defines the VLM architecture:
  SigLIP2 vision encoder (frozen or LoRA) -> MLP projector -> Qwen2.5 LLM (frozen or LoRA)

The design follows the LLaVA-style "unified token space" approach: image patch
embeddings are projected into the LLM's embedding dimension and spliced into
the text token sequence at a reserved <image> placeholder position.
"""

import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor

VISION_MODEL_NAME = "google/siglip2-base-patch16-224"
LLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
IMAGE_TOKEN = "<image>"


class Projector(nn.Module):
    """Maps vision embedding dim -> LLM embedding dim."""

    def __init__(self, vision_dim: int, llm_dim: int, hidden_mult: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim, llm_dim * hidden_mult),
            nn.GELU(),
            nn.Linear(llm_dim * hidden_mult, llm_dim),
        )

    def forward(self, x):
        return self.net(x)


class TinyVLM(nn.Module):
    def __init__(
        self,
        vision_model_name: str = VISION_MODEL_NAME,
        llm_model_name: str = LLM_MODEL_NAME,
        freeze_vision: bool = True,
        freeze_llm: bool = True,
    ):
        super().__init__()

        # --- Vision encoder ---
        self.vision_encoder = AutoModel.from_pretrained(vision_model_name, torch_dtype=torch.float32).vision_model
        self.image_processor = AutoImageProcessor.from_pretrained(vision_model_name)
        vision_dim = self.vision_encoder.config.hidden_size

        # --- LLM ---
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        if IMAGE_TOKEN not in self.tokenizer.get_vocab():
            self.tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
        self.llm = AutoModelForCausalLM.from_pretrained(llm_model_name, torch_dtype=torch.float32)
        self.llm.resize_token_embeddings(len(self.tokenizer))
        llm_dim = self.llm.config.hidden_size

        self.image_token_id = self.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)

        # --- Projector (always trainable) ---
        self.projector = Projector(vision_dim, llm_dim)

        if freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False

        if freeze_llm:
            for p in self.llm.parameters():
                p.requires_grad = False

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """pixel_values: [B, 3, H, W] -> visual tokens [B, num_patches, llm_dim]"""
        vision_out = self.vision_encoder(pixel_values=pixel_values).last_hidden_state
        return self.projector(vision_out)

    def build_inputs_embeds(self, input_ids: torch.Tensor, pixel_values: torch.Tensor):
        """
        Splice visual tokens into the text embedding sequence at the <image>
        placeholder position. Assumes exactly one <image> token per example
        and a single image per example (extend for multi-image later).
        """
        text_embeds = self.llm.get_input_embeddings()(input_ids)  # [B, T, D]
        visual_tokens = self.encode_image(pixel_values)  # [B, N, D]

        batch_embeds = []
        batch_attention_masks = []
        for b in range(input_ids.size(0)):
            ids = input_ids[b]
            img_pos = (ids == self.image_token_id).nonzero(as_tuple=True)[0]
            if len(img_pos) == 0:
                # no image placeholder found -- text-only fallback
                batch_embeds.append(text_embeds[b])
                batch_attention_masks.append(torch.ones(text_embeds.size(1), device=ids.device))
                continue
            pos = img_pos[0].item()
            merged = torch.cat(
                [text_embeds[b, :pos], visual_tokens[b], text_embeds[b, pos + 1:]],
                dim=0,
            )
            batch_embeds.append(merged)
            batch_attention_masks.append(torch.ones(merged.size(0), device=ids.device))

        # pad to max length in batch
        max_len = max(e.size(0) for e in batch_embeds)
        d = batch_embeds[0].size(-1)
        padded = torch.zeros(len(batch_embeds), max_len, d, device=input_ids.device, dtype=batch_embeds[0].dtype)
        attn = torch.zeros(len(batch_embeds), max_len, device=input_ids.device)
        for i, e in enumerate(batch_embeds):
            padded[i, : e.size(0)] = e
            attn[i, : e.size(0)] = 1
        return padded, attn

    def forward(self, input_ids, pixel_values, labels=None):
        inputs_embeds, attention_mask = self.build_inputs_embeds(input_ids, pixel_values)

        # labels must be padded/expanded the same way input was (done in dataset collate_fn)
        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )
        return outputs

    @torch.no_grad()
    def generate(self, input_ids, pixel_values, max_new_tokens=64, **gen_kwargs):
        inputs_embeds, attention_mask = self.build_inputs_embeds(input_ids, pixel_values)
        return self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            **gen_kwargs,
        )


Overwriting model.py


In [20]:
%%writefile dataset.py
"""
dataset.py

Expects a JSONL file where each line is one training example:

  {"image": "images/0001.jpg", "question": "What is the man doing?", "answer": "He is riding a bicycle."}

For Stage 1 (projector pretraining / captioning), just leave "question" empty
or use a fixed prompt like "Describe this image." and put the caption in "answer".

Swap this file's schema for other tasks:
  - Document VQA: image = page render, question = user query, answer = extracted answer
  - OCR: question = "Read the text in this image.", answer = ground truth transcription
  - Classification-as-generation: question = "What category is this?", answer = label
"""

import json
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset

IMAGE_TOKEN = "<image>"


class VQADataset(Dataset):
    def __init__(self, jsonl_path: str, image_root: str, tokenizer, image_processor, max_length: int = 512):
        self.examples = [json.loads(l) for l in Path(jsonl_path).read_text().splitlines() if l.strip()]
        self.image_root = Path(image_root)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.max_length = max_length
        self.image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        image = Image.open(self.image_root / ex["image"]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt")["pixel_values"][0]

        question = ex.get("question", "Describe this image.")
        answer = ex["answer"]

        # Build prompt with chat template so Qwen sees a proper instruction format
        prompt_text = f"{IMAGE_TOKEN} {question}"
        messages = [{"role": "user", "content": prompt_text}]
        rendered_prompt = self.tokenizer.apply_chat_template(
               messages, add_generation_prompt=True, tokenize=False
        )
        prompt_ids = self.tokenizer(rendered_prompt, return_tensors="pt", add_special_tokens=False)["input_ids"][0]

        answer_ids = self.tokenizer(answer + self.tokenizer.eos_token, return_tensors="pt", add_special_tokens=False)[
            "input_ids"
        ][0]

        input_ids = torch.cat([prompt_ids, answer_ids], dim=0)[: self.max_length]

        # labels: mask everything except the answer tokens
        labels = input_ids.clone()
        labels[: prompt_ids.size(0)] = -100

        return {
            "input_ids": input_ids,
            "labels": labels,
            "pixel_values": pixel_values,
            "num_image_tokens": None,  # filled at collate time using model config
        }


def make_collate_fn(model):
    """
    Returns a collate_fn bound to a specific model instance, since we need to
    know how many visual tokens the vision encoder produces per image in
    order to expand the labels tensor to match the spliced sequence length
    that model.forward() will produce internally.
    """
    image_token_id = model.image_token_id
    pad_token_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id

    def collate_fn(batch):
        # Determine num visual tokens dynamically from the vision encoder config
        # (patch_size and image_size come from the processor/config)
        cfg = model.vision_encoder.config
        num_patches = (cfg.image_size // cfg.patch_size) ** 2

        input_ids_list = []
        labels_list = []
        pixel_values_list = []

        for ex in batch:
            ids = ex["input_ids"]
            labels = ex["labels"]

            img_pos = (ids == image_token_id).nonzero(as_tuple=True)[0]
            if len(img_pos) > 0:
                pos = img_pos[0].item()
                # expand labels: image span is always masked (-100), regardless of stage
                labels = torch.cat(
                    [labels[:pos], torch.full((num_patches,), -100, dtype=labels.dtype), labels[pos + 1:]]
                )
                # NOTE: input_ids themselves are NOT expanded here -- model.forward()
                # does the actual embedding splice using pixel_values. We only need
                # input_ids to locate the placeholder, and labels to align with the
                # embedding sequence length after splicing.
            input_ids_list.append(ids)
            labels_list.append(labels)
            pixel_values_list.append(ex["pixel_values"])

        max_ids_len = max(x.size(0) for x in input_ids_list)
        max_labels_len = max(x.size(0) for x in labels_list)

        padded_ids = torch.full((len(batch), max_ids_len), pad_token_id, dtype=torch.long)
        padded_labels = torch.full((len(batch), max_labels_len), -100, dtype=torch.long)

        for i, (ids, labels) in enumerate(zip(input_ids_list, labels_list)):
            padded_ids[i, : ids.size(0)] = ids
            padded_labels[i, : labels.size(0)] = labels

        pixel_values = torch.stack(pixel_values_list)

        return {
            "input_ids": padded_ids,
            "labels": padded_labels,
            "pixel_values": pixel_values,
        }

    return collate_fn


Overwriting dataset.py


In [17]:
%%writefile prepare_data.py
"""
prepare_data.py

Converts two real, public datasets into the JSONL schema used by dataset.py.

Stage 1 -- Flickr8k (captioning)
    Add the Kaggle dataset "adityajn105/flickr8k" as a notebook input.
    It contains:
        Images/*.jpg
        captions.txt   (image,caption rows, 5 captions per image)

Stage 2 -- VQAv2 subset (question answering)
    Pulled from lmms-lab/VQAv2 (Parquet, images embedded, no loading script
    needed -- HuggingFaceM4/VQAv2 relies on a legacy script that `datasets`
    v4+ no longer supports at all). Uses the "validation" split since that's
    the one with ground-truth answers; we take a subset since it's 214k
    examples and more than needed for a fine-tuning demo.
"""

import argparse
import csv
import json
import os
from pathlib import Path

from PIL import Image


def _find_flickr8k_files(kaggle_input_dir: Path):
    """
    Different uploads of the same Flickr8k Kaggle dataset nest files at
    different depths (sometimes Images/ and captions.txt sit directly under
    the input dir, sometimes one level deeper inside a redundant subfolder).
    Search instead of assuming a fixed layout, and fail with a helpful
    listing if nothing is found.
    """
    captions_matches = list(kaggle_input_dir.rglob("captions.txt"))
    # Some uploads use the older Flickr8k naming instead of captions.txt
    if not captions_matches:
        captions_matches = list(kaggle_input_dir.rglob("Flickr8k.token.txt"))

    images_matches = [d for d in kaggle_input_dir.rglob("Images") if d.is_dir()]
    if not images_matches:
        images_matches = [d for d in kaggle_input_dir.rglob("images") if d.is_dir()]

    if not captions_matches or not images_matches:
        tree = "\n".join(str(p) for p in sorted(kaggle_input_dir.rglob("*"))[:50])
        raise FileNotFoundError(
            f"Could not locate captions file and/or Images folder under {kaggle_input_dir}.\n"
            f"First 50 entries found there:\n{tree}\n\n"
            f"Run `!find {kaggle_input_dir} -maxdepth 3` in a Kaggle cell to inspect the "
            f"actual layout, then pass the correct subfolder via --kaggle_input_dir."
        )

    return captions_matches[0], images_matches[0]


def prepare_flickr8k(kaggle_input_dir: str, out_jsonl: str, out_image_dir: str, max_examples: int = None):
    """
    kaggle_input_dir: path to the Flickr8k Kaggle dataset root, e.g.
        /kaggle/input/flickr8k
    Locates captions.txt and the Images/ folder anywhere under this path,
    since different dataset uploads nest them at different depths.
    """
    kaggle_input_dir = Path(kaggle_input_dir)
    captions_file, images_dir = _find_flickr8k_files(kaggle_input_dir)
    print(f"Using captions file: {captions_file}")
    print(f"Using images dir:    {images_dir}")

    os.makedirs(out_image_dir, exist_ok=True)
    out_records = []

    with open(captions_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        f.seek(0)
        # captions.txt has a header row ("image,caption"); the older
        # Flickr8k.token.txt format doesn't -- detect and handle both
        has_header = first_line.strip().lower().startswith("image")
        reader = csv.reader(f)
        if has_header:
            next(reader)
        for row in reader:
            if len(row) < 2:
                continue
            image_name, caption = row[0], ",".join(row[1:]).strip()
            # Flickr8k.token.txt uses "1000268201_693b08cb0e.jpg#0" as the id
            image_name = image_name.split("#")[0]
            src_path = images_dir / image_name
            if not src_path.exists():
                continue

            # symlink instead of copy to save disk space on Kaggle
            dst_path = Path(out_image_dir) / image_name
            if not dst_path.exists():
                try:
                    os.symlink(src_path.resolve(), dst_path)
                except OSError:
                    Image.open(src_path).convert("RGB").save(dst_path)

            out_records.append(
                {"image": image_name, "question": "Describe this image.", "answer": caption}
            )
            if max_examples and len(out_records) >= max_examples:
                break

    if not out_records:
        raise RuntimeError(
            f"Found {captions_file} and {images_dir}, but 0 records matched "
            f"(images referenced in captions weren't found in the images dir). "
            f"Check that the two paths actually correspond to each other."
        )

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for rec in out_records:
            f.write(json.dumps(rec) + "\n")

    print(f"Wrote {len(out_records)} caption examples to {out_jsonl}")


def prepare_vqa_subset(out_jsonl: str, out_image_dir: str, n_examples: int = 5000, split: str = "validation"):
    """
    Pulls a subset of lmms-lab/VQAv2 and materializes it into our JSONL +
    image-folder format. Requires internet access enabled in the Kaggle
    notebook (Settings -> Internet -> On).

    NOTE: we use lmms-lab/VQAv2 rather than HuggingFaceM4/VQAv2. The latter
    ships a legacy dataset-loading script, and the `datasets` library (v4+)
    permanently dropped support for script-based datasets -- there's no
    trust_remote_code workaround anymore. lmms-lab/VQAv2 is a pre-converted
    Parquet mirror with images embedded directly, so it loads with no script.

    lmms-lab/VQAv2 only ships validation/testdev/test splits (no "train"),
    but validation (214k rows) has real ground-truth answers and is plenty
    for a fine-tuning subset -- testdev/test are for competition submission
    and don't include answers.
    """
    from datasets import load_dataset

    os.makedirs(out_image_dir, exist_ok=True)

    ds = load_dataset("lmms-lab/VQAv2", split=split, streaming=True)

    out_records = []
    for i, ex in enumerate(ds):
        if i >= n_examples:
            break

        image = ex["image"].convert("RGB")
        image_name = f"vqa_{i:06d}.jpg"
        image.save(Path(out_image_dir) / image_name)

        question = ex["question"]
        # multiple_choice_answer is already the majority-voted answer;
        # fall back to the first individual annotator answer just in case
        answer = ex.get("multiple_choice_answer") or ex["answers"][0]["answer"]

        out_records.append({"image": image_name, "question": question, "answer": answer})

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for rec in out_records:
            f.write(json.dumps(rec) + "\n")

    print(f"Wrote {len(out_records)} VQA examples to {out_jsonl}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--task", choices=["flickr8k", "vqa"], required=True)
    parser.add_argument("--kaggle_input_dir", default="/kaggle/input/flickr8k")
    parser.add_argument("--out_jsonl", required=True)
    parser.add_argument("--out_image_dir", required=True)
    parser.add_argument("--max_examples", type=int, default=None)
    args = parser.parse_args()

    if args.task == "flickr8k":
        prepare_flickr8k(args.kaggle_input_dir, args.out_jsonl, args.out_image_dir, args.max_examples)
    else:
        prepare_vqa_subset(args.out_jsonl, args.out_image_dir, args.max_examples or 5000)

Overwriting prepare_data.py


In [26]:
%%writefile train_stage1_projector.py
"""
train_stage1_projector.py

Stage 1: freeze vision encoder + LLM, train ONLY the projector on
image-caption pairs. This teaches the projector to map visual features
into the LLM's embedding space.

Usage:
    python train_stage1_projector.py \
        --data data/captions_train.jsonl \
        --image_root data/images \
        --output_dir checkpoints/stage1 \
        --epochs 1 --batch_size 8 --lr 1e-4
"""

import argparse
import os

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from model import TinyVLM
from dataset import VQADataset, make_collate_fn


def main():
    accum_steps = 4
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--image_root", required=True)
    parser.add_argument("--output_dir", default="checkpoints/stage1")
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    model = TinyVLM(freeze_vision=True, freeze_llm=True).to(args.device)
    model.train()

    dataset = VQADataset(args.data, args.image_root, model.tokenizer, model.image_processor)
    collate_fn = make_collate_fn(model)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

    # Only the projector has requires_grad=True at this stage
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=args.lr)

    print(f"Trainable params (should be projector only): {sum(p.numel() for p in trainable_params):,}")

    step = 0
    for epoch in range(args.epochs):
        pbar = tqdm(loader, desc=f"epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(args.device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], pixel_values=batch["pixel_values"], labels=batch["labels"])
            loss = outputs.loss

            loss = loss / accum_steps
            loss.backward()
            if (step + 1) % accum_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
              

            step += 1
            pbar.set_postfix(loss=loss.item())

    torch.save(model.projector.state_dict(), os.path.join(args.output_dir, "projector.pt"))
    print(f"Saved projector weights to {args.output_dir}/projector.pt")


if __name__ == "__main__":
    main()


Overwriting train_stage1_projector.py


In [9]:
%%writefile train_stage2_lora.py
"""
train_stage2_lora.py

Stage 2: load the Stage 1 projector, keep vision encoder frozen, attach LoRA
adapters to the LLM, and train on instruction-style VQA data so the model
learns to actually reason over visual tokens and answer conversationally.

Usage:
    python train_stage2_lora.py \
        --data data/vqa_train.jsonl \
        --image_root data/images \
        --projector_ckpt checkpoints/stage1/projector.pt \
        --output_dir checkpoints/stage2 \
        --epochs 2 --batch_size 4 --lr 2e-4
"""

import argparse
import os

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import LoraConfig, get_peft_model

from model import TinyVLM
from dataset import VQADataset, make_collate_fn


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--image_root", required=True)
    parser.add_argument("--projector_ckpt", required=True)
    parser.add_argument("--output_dir", default="checkpoints/stage2")
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    # Vision stays frozen; LLM base weights frozen too -- LoRA adapters do the learning
    model = TinyVLM(freeze_vision=True, freeze_llm=True)
    model.projector.load_state_dict(torch.load(args.projector_ckpt, map_location="cpu"))
    model.projector.requires_grad_(True)  # keep fine-tuning the projector jointly

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen2 attention proj names
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model.llm = get_peft_model(model.llm, lora_config)
    model.llm.print_trainable_parameters()

    model = model.to(args.device)
    model.train()

    dataset = VQADataset(args.data, args.image_root, model.tokenizer, model.image_processor)
    collate_fn = make_collate_fn(model)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=args.lr)

    for epoch in range(args.epochs):
        pbar = tqdm(loader, desc=f"epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(args.device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], pixel_values=batch["pixel_values"], labels=batch["labels"])
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=loss.item())

    # Save LoRA adapter + projector together
    model.llm.save_pretrained(os.path.join(args.output_dir, "lora_adapter"))
    torch.save(model.projector.state_dict(), os.path.join(args.output_dir, "projector.pt"))
    print(f"Saved LoRA adapter + projector to {args.output_dir}")


if __name__ == "__main__":
    main()


Writing train_stage2_lora.py


In [22]:
%%writefile inference.py
"""
inference.py

Load the base model + trained projector + LoRA adapter, and answer a
question about a given image.

Usage:
    python inference.py \
        --image path/to/image.jpg \
        --question "What is in this image?" \
        --projector_ckpt checkpoints/stage2/projector.pt \
        --lora_dir checkpoints/stage2/lora_adapter
"""

import argparse

import torch
from PIL import Image
from peft import PeftModel

from model import TinyVLM, IMAGE_TOKEN


def load_model(projector_ckpt: str, lora_dir: str = None, device: str = "cuda"):
    model = TinyVLM(freeze_vision=True, freeze_llm=True)
    model.projector.load_state_dict(torch.load(projector_ckpt, map_location="cpu"))

    if lora_dir:
        model.llm = PeftModel.from_pretrained(model.llm, lora_dir)

    model = model.to(device)
    model.eval()
    return model


def answer(model, image_path: str, question: str, device: str = "cuda", max_new_tokens: int = 64) -> str:
    image = Image.open(image_path).convert("RGB")
    pixel_values = model.image_processor(images=image, return_tensors="pt")["pixel_values"].to(device)


    messages = [{"role": "user", "content": f"{IMAGE_TOKEN} {question}"}]
    rendered_prompt = model.tokenizer.apply_chat_template(
          messages, add_generation_prompt=True, tokenize=False
    )
    input_ids = model.tokenizer(rendered_prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # output_ids only contains newly generated tokens since we passed inputs_embeds,
    # not input_ids, to generate() -- decode directly
    return model.tokenizer.decode(output_ids[0], skip_special_tokens=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", required=True)
    parser.add_argument("--question", required=True)
    parser.add_argument("--projector_ckpt", required=True)
    parser.add_argument("--lora_dir", default=None)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    model = load_model(args.projector_ckpt, args.lora_dir, args.device)
    result = answer(model, args.image, args.question, args.device)
    print(result)


Overwriting inference.py


In [10]:
%%writefile push_to_hub.py
"""
push_to_hub.py

Uploads the trained projector + LoRA adapter (+ a model card) to a new
Hugging Face Hub repo.

Prereqs:
    pip install huggingface_hub
    huggingface-cli login          # or set HF_TOKEN env var

Usage:
    python push_to_hub.py \
        --repo_id your-username/tinyvlm-vqa \
        --projector_ckpt checkpoints/stage2/projector.pt \
        --lora_dir checkpoints/stage2/lora_adapter
"""

import argparse
import os
import tempfile
import shutil
from pathlib import Path

from huggingface_hub import HfApi, create_repo

MODEL_CARD_TEMPLATE = """---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B-Instruct
tags:
  - vision-language-model
  - vqa
  - lora
  - siglip2
  - qwen2.5
---

# {repo_name}

A minimal LLaVA-style VLM fine-tuned for visual question answering.

- **Vision encoder:** google/siglip2-base-patch16-224 (frozen)
- **LLM:** Qwen/Qwen2.5-0.5B-Instruct (LoRA fine-tuned)
- **Projector:** 2-layer MLP, trained in Stage 1, refined jointly in Stage 2
- **Task:** Visual question answering

## Files

- `projector.pt` -- projector MLP weights (state_dict)
- `lora_adapter/` -- PEFT LoRA adapter for the Qwen2.5 LLM

## Usage

This repo does not contain a full merged model -- you need the companion
`model.py` / `inference.py` code (from the training repo) to reload the
architecture, then load these weights on top of the base checkpoints:

```python
from model import TinyVLM
from peft import PeftModel
import torch
from huggingface_hub import hf_hub_download, snapshot_download

model = TinyVLM(freeze_vision=True, freeze_llm=True)

projector_path = hf_hub_download(repo_id="{repo_id}", filename="projector.pt")
model.projector.load_state_dict(torch.load(projector_path, map_location="cpu"))

lora_path = snapshot_download(repo_id="{repo_id}", allow_patterns=["lora_adapter/*"])
model.llm = PeftModel.from_pretrained(model.llm, lora_path + "/lora_adapter")
```

## Training data

Stage 1: Flickr8k captions. Stage 2: a subset of VQAv2.

## Limitations

Single image per example, 224x224 input resolution, small (~0.5B) base LLM --
expect casual/simple-VQA quality, not document-grade OCR/reasoning.
"""


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo_id", required=True, help="e.g. your-username/tinyvlm-vqa")
    parser.add_argument("--projector_ckpt", required=True)
    parser.add_argument("--lora_dir", required=True)
    parser.add_argument("--private", action="store_true")
    args = parser.parse_args()

    api = HfApi()
    create_repo(args.repo_id, private=args.private, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        shutil.copy(args.projector_ckpt, tmp / "projector.pt")
        shutil.copytree(args.lora_dir, tmp / "lora_adapter")

        repo_name = args.repo_id.split("/")[-1]
        card = MODEL_CARD_TEMPLATE.format(repo_name=repo_name, repo_id=args.repo_id)
        (tmp / "README.md").write_text(card)

        api.upload_folder(
            folder_path=str(tmp),
            repo_id=args.repo_id,
            commit_message="Upload TinyVLM projector + LoRA adapter",
        )

    print(f"Uploaded to https://huggingface.co/{args.repo_id}")


if __name__ == "__main__":
    main()


Writing push_to_hub.py


## 3. Prepare Stage 1 data -- Flickr8k captions
Uses the Kaggle input dataset added in the sidebar. Adjust the path if your dataset mount differs (check with `!ls /kaggle/input`).

In [11]:
!ls /kaggle/input


datasets


In [14]:
!python prepare_data.py \
    --task flickr8k \
    --kaggle_input_dir /kaggle/input/datasets/adityajn105/flickr8k \
    --out_jsonl /kaggle/working/data/captions_train.jsonl \
    --out_image_dir /kaggle/working/data/images_flickr8k \
    --max_examples 8000


Wrote 8000 caption examples to /kaggle/working/data/captions_train.jsonl


## 4. Prepare Stage 2 data -- VQAv2 subset
Pulled live from the Hugging Face `datasets` library (requires Internet: On). Takes a 5,000-example subset -- adjust `--max_examples` up if you have GPU time to spare.

In [18]:
!python prepare_data.py \
    --task vqa \
    --out_jsonl /kaggle/working/data/vqa_train.jsonl \
    --out_image_dir /kaggle/working/data/images_vqa \
    --max_examples 5000

Resolving data files: 100%|████████████████| 143/143 [00:00<00:00, 33537.55it/s]
Wrote 5000 VQA examples to /kaggle/working/data/vqa_train.jsonl


## 5. Stage 1 -- projector pretraining
Freezes SigLIP2 and Qwen2.5; trains only the projector on Flickr8k captions.

In [27]:
!python train_stage1_projector.py \
    --data /kaggle/working/data/captions_train.jsonl \
    --image_root /kaggle/working/data/images_flickr8k \
    --output_dir /kaggle/working/checkpoints/stage1 \
    --epochs 1 --batch_size 4 --lr 1e-4


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 408/408 [00:00<00:00, 1604.23it/s, Materializing param=
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|█| 290/290 [00:00<00:00, 484.40it/s, Materializing param=m
Trainable params (should be projector only): 1,492,736
epoch 0: 100%|██████████████████| 2000/2000 [25:05<00:00,  1.33it/s, loss=0.786]
Saved projector weights to /kaggle/working/checkpoints/stage1/projector.pt


## 6. Stage 2 -- LoRA instruction tuning
Loads the Stage 1 projector, attaches LoRA to Qwen2.5, trains on the VQAv2 subset.

In [30]:
!python train_stage2_lora.py \
    --data /kaggle/working/data/vqa_train.jsonl \
    --image_root /kaggle/working/data/images_vqa \
    --projector_ckpt /kaggle/working/checkpoints/stage1/projector.pt \
    --output_dir /kaggle/working/checkpoints/stage2 \
    --epochs 2 --batch_size 8 --lr 2e-4


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 408/408 [00:00<00:00, 1812.22it/s, Materializing param=
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|█| 290/290 [00:00<00:00, 490.52it/s, Materializing param=m
trainable params: 2,162,688 || all params: 495,953,536 || trainable%: 0.4361
epoch 1: 100%|█████████████████████| 625/625 [15:26<00:00,  1.48s/it, loss=1.34]
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  wa

In [29]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.9 MB/s eta 0:00:0000:0100:01


## 7. Quick inference sanity check
Grab one image from the VQA subset and ask it a question.

In [ ]:
import json, glob

sample = json.loads(open('/kaggle/working/data/vqa_train.jsonl').readline())
image_path = f"/kaggle/working/data/images_vqa/{sample['image']}"
print('Question:', sample['question'])
print('Ground truth:', sample['answer'])

!python inference.py \
    --image "{image_path}" \
    --question "{sample['question']}" \
    --projector_ckpt /kaggle/working/checkpoints/stage2/projector.pt \
    --lora_dir /kaggle/working/checkpoints/stage2/lora_adapter


## 8. Push weights to the Hugging Face Hub

Add your HF token as a Kaggle Secret (`Add-ons -> Secrets`, name it `HF_TOKEN`),
or run `huggingface-cli login` interactively and paste a token from
https://huggingface.co/settings/tokens (needs write access).

Replace `your-username/tinyvlm-vqa` with your actual namespace/repo name.


In [36]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No Kaggle secret named HF_TOKEN found -- run the login cell below instead.")


In [37]:
from huggingface_hub import login
import os

if "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"])
else:
    login()  # will prompt for a token interactively


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [38]:
!python push_to_hub.py \
    --repo_id dhruvpatel93/tinyvlm-vqa \
    --projector_ckpt /kaggle/working/checkpoints/stage2/projector.pt \
    --lora_dir /kaggle/working/checkpoints/stage2/lora_adapter


Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|▌             | 40.0MB / 1.10GB            

Processing Files (0 / 1)      :   4%|▌             | 40.0MB / 1.10GB,  199MB/s  

  ...adapter_model.safetensors:   4%|▌             | 40.0MB / 1.10GB            

Processing Files (0 / 1)      :  16%|██▏           |  176MB / 1.10GB,  293MB/s  

Processing Files (0 / 1)      :  31%|████▎         |  344MB / 1.10GB,  430MB/s  

Processing Files (0 / 1)      :  47%|██████▌       |  520MB / 1.10GB,  520MB/s  

Processing Files (0 / 1)      :  63%|████████▊     |  696MB / 1.10GB,  580MB/s  

Processing Files (0 / 1)      :  79%|███████████   |  872MB / 1.10GB,  623MB/s  

Processing Files (0 / 1)      :  94%|█████████████▏| 1.04GB / 1.10GB,  650MB/s  


  .../tmpzx7jd8y2/projector.pt:   4%|▌             |  228kB / 5.97MB            

  ...adapter_mod